# 🏙️ CivicPulse Sight — Where Should We Fix First?
## Full Pipeline: EDA → Outlier Detection → Preprocessing → Regularized Model

> **Problem**: Cities receive reports about road damage, traffic, flooding, streetlights, drainage,
> and pedestrian safety. Budgets and field teams are limited. Authorities need a transparent way to
> combine incident history, public complaints, and local context to decide which locations deserve attention first.

**Datasets** (joined on `location_id`):
- `locations.csv` — 1,000 locations with spatial/contextual attributes
- `incidents.csv` — 10,000 incident records (type, severity, resolution)
- `public_complaints.csv` — 8,000 complaint records (category, priority, status)

**Goal**: Build a severity-and-frequency-based **priority score** per location and train a regularized regression model.
**Engine**: CivicPulse Sight Urban Intelligence Platform.

---
## 📦 Section 0 — Imports & Style Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import zscore
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#1a1d2e",
    "axes.edgecolor": "#2e3250", "axes.labelcolor": "#c8cdd8",
    "xtick.color": "#8890a8", "ytick.color": "#8890a8",
    "text.color": "#c8cdd8", "grid.color": "#2e3250",
    "grid.linestyle": "--", "grid.alpha": 0.5, "figure.dpi": 110,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlecolor": "#e8ecf5", "axes.labelsize": 11,
    "legend.facecolor": "#1a1d2e", "legend.edgecolor": "#2e3250",
})

ACCENT  = "#6c63ff"
ACCENT2 = "#ff6584"
ACCENT3 = "#43e97b"
ACCENT4 = "#f7971e"
CMAP_HOT = LinearSegmentedColormap.from_list("hot_city", ["#1a1d2e", "#f7971e", "#c62a47"])
DATA_DIR = "../03_smart_city_where_fix_first/"
SEV_W     = {"Low": 1, "Medium": 2, "High": 4, "Critical": 8}
SEV_ORDER = ["Low", "Medium", "High", "Critical"]
SEV_COLORS= ["#43e97b", "#f7971e", "#ff6584", "#c62a47"]
RC_ORDER  = ["Good", "Fair", "Poor", "Critical"]
RC_COLORS = ["#43e97b", "#f7971e", "#ff6584", "#c62a47"]
print("✅ Setup complete.")

---
## 📂 Section 1 — Load Raw Data

In [ ]:
locations  = pd.read_csv(DATA_DIR + "locations.csv")
incidents  = pd.read_csv(DATA_DIR + "incidents.csv")
complaints = pd.read_csv(DATA_DIR + "public_complaints.csv")
incidents["timestamp"]    = pd.to_datetime(incidents["timestamp"],    errors="coerce")
complaints["created_at"]  = pd.to_datetime(complaints["created_at"],  errors="coerce")
print(f"locations  : {locations.shape}")
print(f"incidents  : {incidents.shape}")
print(f"complaints : {complaints.shape}")
locations.head(3)

---
## 📊 Section 2 — Exploratory Data Analysis

We visualise every dimension of the raw data before touching it.

### 2.1 Missing Values per Dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Dataset — Missing Value Analysis", fontsize=15, fontweight="bold")
for ax, (name, df_tmp, col) in zip(axes, [
        ("locations",  locations,  ACCENT),
        ("incidents",  incidents,  ACCENT2),
        ("complaints", complaints, ACCENT3)]):
    null_pct = df_tmp.isnull().sum() / len(df_tmp) * 100
    bars = ax.barh(null_pct.index, null_pct.values, color=col, alpha=0.85, height=0.6)
    ax.set_title(f"{name}\n{df_tmp.shape[0]:,} rows, {df_tmp.shape[1]} cols", color=col)
    ax.set_xlabel("Missing %")
    ax.set_xlim(0, max(null_pct.max() + 2, 5))
    for b, v in zip(bars, null_pct.values):
        ax.text(v + 0.1, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center", fontsize=8)
    ax.grid(True, axis="x")
plt.tight_layout()
plt.savefig("2_1_missing.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("No missing values found across all datasets!")

### 2.2 Incident Type, Severity & Reporter

In [ ]:
wc = ["#6c63ff","#ff6584","#43e97b","#f7971e","#a78bfa","#34d399"]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Incidents — Type, Severity & Reporter", fontsize=14, fontweight="bold")
tc = incidents["incident_type"].value_counts()
axes[0].pie(tc, labels=tc.index, autopct="%1.1f%%", colors=wc, startangle=140,
    textprops={"color": "#c8cdd8", "fontsize": 9},
    wedgeprops={"edgecolor": "#0f1117", "linewidth": 1.5})
axes[0].set_title("Incident Types")
sc = incidents["severity"].value_counts().reindex(SEV_ORDER).fillna(0)
bars = axes[1].bar(sc.index, sc.values, color=SEV_COLORS, width=0.6, edgecolor="#0f1117")
for b in bars:
    h = b.get_height()
    if not np.isnan(h) and h > 0:
        axes[1].text(b.get_x()+b.get_width()/2, h+30,
            f"{int(h):,}", ha="center", fontsize=9)
axes[1].set_title("Severity Counts")
axes[1].set_ylabel("Count")
axes[1].grid(True, axis="y")
rc = incidents["reported_by"].value_counts()
axes[2].barh(rc.index, rc.values,
    color=[ACCENT, ACCENT2, ACCENT3, ACCENT4], height=0.5)
for i, v in enumerate(rc.values):
    axes[2].text(v + 30, i, f"{v:,}", va="center", fontsize=9)
axes[2].set_title("Reported By")
axes[2].set_xlabel("Count")
axes[2].grid(True, axis="x")
plt.tight_layout()
plt.savefig("2_2_incidents.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 2.3 Complaints — Category, Priority & Status

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Public Complaints — Category, Priority & Status", fontsize=14, fontweight="bold")
cat = complaints["category"].value_counts()
wc  = ["#6c63ff","#ff6584","#43e97b","#f7971e","#a78bfa","#34d399"]
axes[0].barh(cat.index, cat.values, color=wc, height=0.55)
for i, v in enumerate(cat.values):
    axes[0].text(v + 10, i, f"{v:,}", va="center", fontsize=9)
axes[0].set_title("Complaint Categories")
axes[0].set_xlabel("Count")
axes[0].grid(True, axis="x")
pri = complaints["priority_reported"].value_counts()
pc  = {"High": ACCENT2, "Normal": ACCENT, "Low": ACCENT3}
axes[1].pie(pri, labels=pri.index, autopct="%1.1f%%",
    colors=[pc.get(p, ACCENT) for p in pri.index], startangle=90,
    textprops={"color": "#c8cdd8", "fontsize": 10},
    wedgeprops={"edgecolor": "#0f1117", "linewidth": 1.5})
axes[1].set_title("Priority Reported")
st  = complaints["status"].value_counts()
sc2 = ["#43e97b","#6c63ff","#f7971e","#ff6584","#a78bfa"]
axes[2].bar(st.index, st.values, color=sc2[:len(st)], width=0.55, edgecolor="#0f1117")
for b in axes[2].patches:
    h = b.get_height()
    if not np.isnan(h) and h > 0:
        axes[2].text(b.get_x()+b.get_width()/2, h+10,
            f"{int(h):,}", ha="center", fontsize=8)
axes[2].set_title("Complaint Status")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=15)
axes[2].grid(True, axis="y")
plt.tight_layout()
plt.savefig("2_3_complaints.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 2.4 Location Attribute Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Location Attributes", fontsize=14, fontweight="bold")
rt = locations["road_type"].value_counts()
axes[0,0].bar(rt.index, rt.values,
    color=["#6c63ff","#ff6584","#43e97b","#f7971e","#a78bfa"],
    width=0.6, edgecolor="#0f1117")
axes[0,0].set_title("Road Type")
axes[0,0].tick_params(axis="x", rotation=15)
axes[0,0].grid(True, axis="y")
rc2 = locations["road_condition"].value_counts().reindex(RC_ORDER).fillna(0)
axes[0,1].bar(RC_ORDER, rc2.values, color=RC_COLORS, width=0.6, edgecolor="#0f1117")
axes[0,1].set_title("Road Condition")
axes[0,1].grid(True, axis="y")
pa = locations["pedestrian_activity"].value_counts()
axes[0,2].bar(pa.index, pa.values, color=[ACCENT3, ACCENT, ACCENT2], width=0.5, edgecolor="#0f1117")
axes[0,2].set_title("Pedestrian Activity")
axes[0,2].grid(True, axis="y")
axes[1,0].hist(locations["population_density_per_km2"], bins=30,
    color=ACCENT, edgecolor="#0f1117", alpha=0.85)
axes[1,0].set_title("Population Density Distribution")
axes[1,0].set_xlabel("people/km2")
axes[1,0].grid(True, axis="y")
prox = pd.DataFrame({
    "Near School":   locations["near_school"].value_counts().reindex([0,1]).fillna(0).values,
    "Near Hospital": locations["near_hospital"].value_counts().reindex([0,1]).fillna(0).values},
    index=["No","Yes"])
prox.plot(kind="bar", ax=axes[1,1], color=[ACCENT2, ACCENT3],
    width=0.5, edgecolor="#0f1117", rot=0)
axes[1,1].set_title("Proximity to School / Hospital")
axes[1,1].set_ylabel("Count")
axes[1,1].legend(facecolor="#1a1d2e")
axes[1,1].grid(True, axis="y")
sc = axes[1,2].scatter(locations["longitude"], locations["latitude"],
    c=locations["population_density_per_km2"], cmap=CMAP_HOT, s=15, alpha=0.7)
plt.colorbar(sc, ax=axes[1,2], label="Population Density")
axes[1,2].set_title("Geographic Spread of Locations")
axes[1,2].set_xlabel("Longitude")
axes[1,2].set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("2_4_locations.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 2.5 Time-Series Trends

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.suptitle("Temporal Trends — Incidents & Complaints", fontsize=14, fontweight="bold")
im = incidents.set_index("timestamp").resample("ME")["incident_id"].count()
axes[0].fill_between(im.index, im.values, color=ACCENT, alpha=0.3)
axes[0].plot(im.index, im.values, color=ACCENT, linewidth=2)
axes[0].set_title("Monthly Incident Count")
axes[0].set_ylabel("Incidents")
axes[0].grid(True)
itm = (incidents.set_index("timestamp")
    .groupby([pd.Grouper(freq="ME"), "incident_type"])["incident_id"]
    .count().unstack(fill_value=0))
tc2 = ["#6c63ff","#ff6584","#43e97b","#f7971e","#a78bfa","#34d399"]
for col, color in zip(itm.columns, tc2):
    axes[1].plot(itm.index, itm[col], label=col, color=color, linewidth=1.8, alpha=0.9)
axes[1].set_title("Monthly Incidents by Type")
axes[1].set_ylabel("Count")
axes[1].legend(loc="upper left", fontsize=8)
axes[1].grid(True)
plt.tight_layout()
plt.savefig("2_5_timeseries.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 2.6 Resolution Time Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Resolution Time Distributions", fontsize=14, fontweight="bold")
axes[0].hist(incidents["resolution_hours"].clip(upper=200), bins=50,
    color=ACCENT, edgecolor="#0f1117", alpha=0.85)
axes[0].axvline(incidents["resolution_hours"].median(), color=ACCENT2,
    linestyle="--", linewidth=2,
    label=f'Median: {incidents["resolution_hours"].median():.1f}h')
axes[0].set_title("Incident Resolution Hours (clipped at 200)")
axes[0].set_xlabel("Hours"); axes[0].set_ylabel("Count")
axes[0].legend(); axes[0].grid(True, axis="y")
axes[1].hist(complaints["days_to_resolution"].clip(upper=60), bins=50,
    color=ACCENT3, edgecolor="#0f1117", alpha=0.85)
axes[1].axvline(complaints["days_to_resolution"].median(), color=ACCENT2,
    linestyle="--", linewidth=2,
    label=f'Median: {complaints["days_to_resolution"].median():.1f}d')
axes[1].set_title("Complaint Days to Resolution (clipped at 60)")
axes[1].set_xlabel("Days"); axes[1].legend(); axes[1].grid(True, axis="y")
plt.tight_layout()
plt.savefig("2_6_resolution.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(incidents["resolution_hours"].describe())
print(complaints["days_to_resolution"].describe())

---
## 🔧 Section 3 — Feature Engineering (Aggregate per Location)

In [ ]:
incidents["sev_weight"] = incidents["severity"].map(SEV_W)
inc_agg = incidents.groupby("location_id").agg(
    total_incidents      = ("incident_id",   "count"),
    total_injuries       = ("injuries",       "sum"),
    avg_resolution_hours = ("resolution_hours","mean"),
    max_resolution_hours = ("resolution_hours","max"),
    weighted_severity    = ("sev_weight",     "sum"),
    pct_resolved         = ("resolved",       "mean"),
    critical_count       = ("severity",       lambda x: (x=="Critical").sum()),
    high_count           = ("severity",       lambda x: (x=="High").sum()),
).reset_index()
PRI_W = {"Low": 1, "Normal": 2, "High": 4}
complaints["pri_weight"] = complaints["priority_reported"].map(PRI_W)
comp_agg = complaints.groupby("location_id").agg(
    total_complaints    = ("complaint_id",       "count"),
    avg_days_resolution = ("days_to_resolution", "mean"),
    weighted_priority   = ("pri_weight",         "sum"),
    high_priority_count = ("priority_reported",  lambda x: (x=="High").sum()),
    open_complaints     = ("status",             lambda x: (x=="Open").sum()),
).reset_index()
df = (locations.merge(inc_agg,  on="location_id", how="left")
               .merge(comp_agg, on="location_id", how="left"))
df.fillna(0, inplace=True)
print(f"Merged dataset: {df.shape}")
df.head(3)

### 3.1 Priority Score Definition (Target Variable)

```
priority_score = 0.40 × norm(weighted_severity)
              + 0.25 × norm(total_incidents)
              + 0.15 × norm(weighted_priority)   ← from complaints
              + 0.10 × norm(avg_resolution_hours)
              + 0.10 × norm(high_priority_count)
```

In [ ]:
def minmax(s):
    r = s.max() - s.min()
    return (s - s.min()) / r if r > 0 else s * 0

df["priority_score"] = (
    0.40 * minmax(df["weighted_severity"])    +
    0.25 * minmax(df["total_incidents"])      +
    0.15 * minmax(df["weighted_priority"])    +
    0.10 * minmax(df["avg_resolution_hours"]) +
    0.10 * minmax(df["high_priority_count"])
)
print("Priority score stats:")
print(df["priority_score"].describe())
top10 = df.nlargest(10,"priority_score")[["location_id","zone","road_condition",
                                           "total_incidents","critical_count","priority_score"]]
print("\nTop 10 locations:")
print(top10.to_string(index=False))

### 3.2 Priority Score Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Priority Score Distribution", fontsize=14, fontweight="bold")
axes[0].hist(df["priority_score"], bins=40, color=ACCENT, edgecolor="#0f1117", alpha=0.85)
axes[0].axvline(df["priority_score"].mean(),   color=ACCENT2, linewidth=2, linestyle="--",
    label=f'Mean={df["priority_score"].mean():.3f}')
axes[0].axvline(df["priority_score"].median(), color=ACCENT3, linewidth=2, linestyle="-.",
    label=f'Median={df["priority_score"].median():.3f}')
axes[0].set_title("Priority Score Histogram")
axes[0].set_xlabel("Score"); axes[0].set_ylabel("# Locations")
axes[0].legend(); axes[0].grid(True, axis="y")
rcp = df.groupby("road_condition")["priority_score"].mean().reindex(RC_ORDER).fillna(0)
axes[1].bar(RC_ORDER, rcp.values, color=RC_COLORS, width=0.5, edgecolor="#0f1117")
axes[1].set_title("Avg Priority by Road Condition")
axes[1].set_ylabel("Avg Score"); axes[1].grid(True, axis="y")
sc = axes[2].scatter(df["longitude"], df["latitude"], c=df["priority_score"],
    cmap=CMAP_HOT, s=30, alpha=0.8, edgecolors="none")
plt.colorbar(sc, ax=axes[2], label="Priority Score")
axes[2].set_title("Geographic Priority Heatmap")
axes[2].set_xlabel("Longitude"); axes[2].set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("3_2_priority.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

---
## 🔎 Section 4 — Outlier Detection

Three complementary approaches:
1. **Box plots** — visual detection across all numeric features
2. **Z-score** (|z| > 3) — parametric statistical method
3. **IQR rule** (1.5×IQR) — robust non-parametric method

In [ ]:
NUMERIC = [
    "population_density_per_km2", "total_incidents", "total_injuries",
    "avg_resolution_hours", "max_resolution_hours", "weighted_severity",
    "pct_resolved", "critical_count", "high_count", "total_complaints",
    "avg_days_resolution", "weighted_priority", "high_priority_count", "open_complaints"
]
df[NUMERIC].describe().round(2)

### 4.1 Box Plots — Visual Outlier Detection

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
fig.suptitle("Box Plots — Raw Numeric Features", fontsize=14, fontweight="bold")
axes = axes.flatten()
bx_colors = [ACCENT, ACCENT2, ACCENT3, ACCENT4, "#a78bfa"] * 3
for i, col in enumerate(NUMERIC):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
        boxprops=dict(facecolor=bx_colors[i], alpha=0.6, color="#c8cdd8"),
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(color="#8890a8"), capprops=dict(color="#8890a8"),
        flierprops=dict(marker="o", color=ACCENT2,
            markerfacecolor=ACCENT2, markersize=3, alpha=0.6))
    axes[i].set_title(col.replace("_", " "), fontsize=8)
    axes[i].grid(True, axis="y")
for j in range(len(NUMERIC), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig("4_1_boxplots.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 4.2 Z-Score Method (|z| > 3)

In [ ]:
z      = np.abs(zscore(df[NUMERIC]))
ZTHR   = 3.0
z_any  = (z > ZTHR).any(axis=1)
print(f"Z-score outlier rows: {z_any.sum()} ({z_any.mean()*100:.1f}%)")
z_col  = (z > ZTHR).sum(axis=0)
zdf    = pd.DataFrame({"column": NUMERIC, "z_outliers": z_col})\
           .sort_values("z_outliers", ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(zdf["column"], zdf["z_outliers"], color=ACCENT2, alpha=0.85, height=0.6)
for b, v in zip(bars, zdf["z_outliers"]):
    if v > 0:
        ax.text(v + 0.2, b.get_y() + b.get_height()/2, str(v), va="center", fontsize=9)
ax.set_title(f"Z-Score Outliers per Feature (|z|>{ZTHR})", color="#e8ecf5",
    fontsize=13, fontweight="bold")
ax.set_xlabel("Count")
ax.grid(True, axis="x")
plt.tight_layout()
plt.savefig("4_2_zscore.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 4.3 IQR Method (1.5×IQR Rule)

In [ ]:
def iqr_flag(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    return (s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)

iqr_flags = df[NUMERIC].apply(iqr_flag)
iqr_any   = iqr_flags.any(axis=1)
print(f"IQR outlier rows: {iqr_any.sum()} ({iqr_any.mean()*100:.1f}%)")
idf = iqr_flags.sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(idf.index, idf.values, color=ACCENT3, alpha=0.85, height=0.6)
for b, v in zip(bars, idf.values):
    if v > 0:
        ax.text(v + 0.2, b.get_y() + b.get_height()/2, str(v), va="center", fontsize=9)
ax.set_title("IQR Outliers per Feature (1.5×IQR rule)", color="#e8ecf5",
    fontsize=13, fontweight="bold")
ax.set_xlabel("Count")
ax.grid(True, axis="x")
plt.tight_layout()
plt.savefig("4_3_iqr.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 4.4 Scatter — Outlier Highlighting (Z-score flagged in red)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Outlier Scatter Plots (Z-score flagged in red)", fontsize=14, fontweight="bold")
pairs = [
    ("total_incidents",    "weighted_severity"),
    ("avg_resolution_hours","total_incidents"),
    ("total_complaints",   "weighted_priority"),
]
for ax, (xc, yc) in zip(axes, pairs):
    ax.scatter(df.loc[~z_any, xc], df.loc[~z_any, yc],
        color=ACCENT, alpha=0.4, s=18, label="Normal")
    ax.scatter(df.loc[z_any,  xc], df.loc[z_any,  yc],
        color=ACCENT2, alpha=0.85, s=40, label="Outlier", marker="X")
    ax.set_xlabel(xc.replace("_"," "))
    ax.set_ylabel(yc.replace("_"," "))
    ax.set_title(f"{xc.replace(chr(95)," ")} vs {yc.replace(chr(95)," ")}")
    ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
plt.savefig("4_4_outlier_scatter.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

---
## 🧹 Section 5 — Preprocessing

**Strategy**: Winsorize (cap at 1st–99th percentile) to preserve all locations while limiting extreme influence.
Removing outlier rows is avoided because extreme locations ARE the high-priority ones we want to find.

### 5.1 Winsorizing (Cap at 1st–99th Percentile)

In [ ]:
df_c = df.copy()
cap_cols = ["total_incidents","total_injuries","avg_resolution_hours",
            "max_resolution_hours","weighted_severity","critical_count","high_count",
            "total_complaints","avg_days_resolution","weighted_priority",
            "high_priority_count","open_complaints"]
for col in cap_cols:
    lo = df_c[col].quantile(0.01)
    hi = df_c[col].quantile(0.99)
    df_c[col] = df_c[col].clip(lower=lo, upper=hi)
print("Winsorizing applied (1st–99th percentile)")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Before vs After Winsorizing — avg_resolution_hours",
    fontsize=14, fontweight="bold")
axes[0].hist(df["avg_resolution_hours"],   bins=40, color=ACCENT2, edgecolor="#0f1117", alpha=0.8)
axes[0].set_title("BEFORE — Raw"); axes[0].set_xlabel("Hours"); axes[0].grid(True, axis="y")
axes[1].hist(df_c["avg_resolution_hours"], bins=40, color=ACCENT3, edgecolor="#0f1117", alpha=0.8)
axes[1].set_title("AFTER — Winsorized"); axes[1].set_xlabel("Hours"); axes[1].grid(True, axis="y")
plt.tight_layout()
plt.savefig("5_1_winsorize.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 5.2 Categorical Encoding

- **Ordinal**: `road_condition` (Good→0 … Critical→3), `pedestrian_activity` (Low→0 … High→2)
- **One-hot**: `zone` (20 levels), `road_type` (5 levels)

In [ ]:
df_c["road_condition_enc"] = df_c["road_condition"].map({"Good":0,"Fair":1,"Poor":2,"Critical":3})
df_c["pedestrian_enc"]     = df_c["pedestrian_activity"].map({"Low":0,"Medium":1,"High":2})
dummies = pd.get_dummies(df_c[["zone","road_type"]], drop_first=True)
df_c    = pd.concat([df_c, dummies], axis=1)
print(f"Ordinal encoded: road_condition_enc, pedestrian_enc")
print(f"One-hot zone cols   : {df_c.filter(like="zone_").shape[1]}")
print(f"One-hot road_type   : {df_c.filter(like="road_type_").shape[1]}")

### 5.3 Feature Matrix & Correlation Heatmap

In [ ]:
FEAT_COLS = (
    ["population_density_per_km2","near_school","near_hospital",
     "road_condition_enc","pedestrian_enc","total_incidents","total_injuries",
     "avg_resolution_hours","max_resolution_hours","weighted_severity","pct_resolved",
     "critical_count","high_count","total_complaints","avg_days_resolution",
     "weighted_priority","high_priority_count","open_complaints"]
    + list(df_c.filter(like="zone_").columns)
    + list(df_c.filter(like="road_type_").columns)
)
X = df_c[FEAT_COLS].copy()
y = (
    0.40 * minmax(df_c["weighted_severity"])    +
    0.25 * minmax(df_c["total_incidents"])      +
    0.15 * minmax(df_c["weighted_priority"])    +
    0.10 * minmax(df_c["avg_resolution_hours"]) +
    0.10 * minmax(df_c["high_priority_count"])
)
df_c["priority_score_w"] = y
print(f"Feature matrix: {X.shape}, Target y range [{y.min():.3f}, {y.max():.3f}]")

In [ ]:
CORE = ["population_density_per_km2","total_incidents","total_injuries",
        "avg_resolution_hours","weighted_severity","pct_resolved",
        "critical_count","total_complaints","weighted_priority",
        "high_priority_count","road_condition_enc","pedestrian_enc"]
corr = df_c[CORE + ["priority_score_w"]].corr()
cmap_div = LinearSegmentedColormap.from_list("div", ["#c62a47","#1a1d2e","#6c63ff"])
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, cmap=cmap_div, center=0, annot=True, fmt=".2f",
    linewidths=0.5, annot_kws={"size":8,"color":"#e8ecf5"},
    linecolor="#0f1117", cbar_kws={"shrink":0.8}, ax=ax)
ax.set_title("Correlation Matrix — Core Features + Priority Score",
    fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("5_3_correlation.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 5.4 Train / Test Split & RobustScaler

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler  = RobustScaler()
Xtr     = scaler.fit_transform(X_train)
Xte     = scaler.transform(X_test)
print(f"Train: {Xtr.shape}, Test: {Xte.shape}")
fi = list(X.columns).index("weighted_severity")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("weighted_severity — Before vs After RobustScaler",
    fontsize=13, fontweight="bold")
axes[0].hist(X_train["weighted_severity"], bins=35, color=ACCENT,  edgecolor="#0f1117", alpha=0.8)
axes[0].set_title("Before Scaling"); axes[0].grid(True, axis="y")
axes[1].hist(Xtr[:, fi],               bins=35, color=ACCENT3, edgecolor="#0f1117", alpha=0.8)
axes[1].set_title("After RobustScaler"); axes[1].grid(True, axis="y")
plt.tight_layout()
plt.savefig("5_4_scaling.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

---
## 🏗️ Section 6 — Architectural Approach & Regularized Models

```
Raw CSVs → Aggregate per location_id → Feature Engineering
                                              │
              Winsorize (1-99%)               │
              Ordinal / One-hot Encode        │
              RobustScaler                    │
                    │
              80 / 20 Split
                    │
     ┌──────────────┼──────────────┐
   Ridge       ElasticNet        Lasso
   (L2)        (L1+L2)           (L1)
     └──────────────┼──────────────┘
               5-Fold CV
                    │
     Best alpha via regularization path search
                    │
     Feature importance via coefficients
```

### 6.1 Train Baseline + Regularized Models

In [ ]:
models = {
    "OLS (Baseline)": LinearRegression(),
    "Ridge (L2)":     Ridge(alpha=1.0),
    "Lasso (L1)":     Lasso(alpha=0.001, max_iter=10000),
    "ElasticNet":     ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, model in models.items():
    cv_r = np.sqrt(-cross_val_score(model, Xtr, y_train, cv=kf,
                   scoring="neg_mean_squared_error"))
    model.fit(Xtr, y_train)
    yp = model.predict(Xte)
    results[name] = {
        "cv_mean": cv_r.mean(), "cv_std": cv_r.std(),
        "rmse":    np.sqrt(mean_squared_error(y_test, yp)),
        "mae":     mean_absolute_error(y_test, yp),
        "r2":      r2_score(y_test, yp),
        "yp":      yp, "model": model
    }
    print(f"{name:22s}| CV RMSE={cv_r.mean():.4f}±{cv_r.std():.4f}"
          f" | R2={r2_score(y_test,yp):.4f} | RMSE={np.sqrt(mean_squared_error(y_test,yp)):.4f}")
MN = list(results.keys())
mc = ["#f7971e","#6c63ff","#ff6584","#43e97b"]

### 6.2 Model Comparison — CV RMSE, R², MAE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Model Comparison — Regularized vs Baseline", fontsize=14, fontweight="bold")
bars = axes[0].bar(MN, [results[m]["cv_mean"] for m in MN],
    yerr=[results[m]["cv_std"] for m in MN],
    color=mc, width=0.5, edgecolor="#0f1117",
    capsize=5, error_kw={"ecolor":"white","linewidth":1.5})
for b, v in zip(bars, [results[m]["cv_mean"] for m in MN]):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
        f"{v:.4f}", ha="center", fontsize=8)
axes[0].set_title("5-Fold CV RMSE"); axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=20); axes[0].grid(True, axis="y")
axes[1].bar(MN, [results[m]["r2"] for m in MN], color=mc, width=0.5, edgecolor="#0f1117")
axes[1].set_title("Test R2 Score"); axes[1].set_ylabel("R2")
axes[1].tick_params(axis="x", rotation=20); axes[1].grid(True, axis="y")
axes[2].bar(MN, [results[m]["mae"] for m in MN], color=mc, width=0.5, edgecolor="#0f1117")
axes[2].set_title("Test MAE"); axes[2].set_ylabel("MAE")
axes[2].tick_params(axis="x", rotation=20); axes[2].grid(True, axis="y")
plt.tight_layout()
plt.savefig("6_2_model_compare.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 6.3 Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Predicted vs Actual Priority Score", fontsize=14, fontweight="bold")
for ax, (name, res), color in zip(axes.flatten(), results.items(), mc):
    yp = res["yp"]
    ax.scatter(y_test, yp, color=color, alpha=0.5, s=20, edgecolors="none")
    lim = [min(y_test.min(), yp.min())-0.02, max(y_test.max(), yp.max())+0.02]
    ax.plot(lim, lim, "--", color="white", linewidth=1.5, label="Perfect")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
    ax.set_title(f"{name}  |  R2={res["r2"]:.4f}")
    ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
plt.savefig("6_3_pred_actual.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 6.4 Residual Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Residual Plots", fontsize=14, fontweight="bold")
for ax, (name, res), color in zip(axes.flatten(), results.items(), mc):
    resid = y_test.values - res["yp"]
    ax.scatter(res["yp"], resid, color=color, alpha=0.5, s=15)
    ax.axhline(0, color="white", linewidth=1.5, linestyle="--")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
    ax.set_title(name); ax.grid(True)
plt.tight_layout()
plt.savefig("6_4_residuals.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 6.5 Ridge Coefficient Importance (Top 20)

In [ ]:
rm      = results["Ridge (L2)"]["model"]
coef_df = pd.DataFrame({"feature": FEAT_COLS, "coef": rm.coef_})
coef_df = coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index).head(20)
fig, ax = plt.subplots(figsize=(12, 7))
cols    = [ACCENT3 if v > 0 else ACCENT2 for v in coef_df["coef"]]
ax.barh(coef_df["feature"][::-1], coef_df["coef"][::-1],
    color=cols[::-1], height=0.6, edgecolor="#0f1117")
ax.axvline(0, color="white", linewidth=1)
ax.set_title("Ridge — Top 20 Feature Coefficients",
    fontsize=13, fontweight="bold", color="#e8ecf5")
ax.set_xlabel("Coefficient Value")
ax.grid(True, axis="x")
gp = mpatches.Patch(color=ACCENT3, label="Positive (↑ score)")
rp = mpatches.Patch(color=ACCENT2, label="Negative (↓ score)")
ax.legend(handles=[gp, rp])
plt.tight_layout()
plt.savefig("6_5_ridge_coefs.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 6.6 Lasso Sparsity Analysis

In [ ]:
lm  = results["Lasso (L1)"]["model"]
lc  = pd.Series(lm.coef_, index=FEAT_COLS)
nz  = (lc != 0).sum(); zo = (lc == 0).sum()
print(f"Lasso: {nz} features kept, {zo} zeroed out of {len(FEAT_COLS)}")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Lasso (L1) Sparsity Analysis", fontsize=14, fontweight="bold")
axes[0].pie([nz, zo], labels=[f"Kept ({nz})", f"Zeroed ({zo})"],
    colors=[ACCENT3, ACCENT2], autopct="%1.0f%%", startangle=90,
    textprops={"color":"#c8cdd8"},
    wedgeprops={"edgecolor":"#0f1117","linewidth":1.5})
axes[0].set_title("Feature Selection by Lasso")
lt = lc[lc != 0].abs().sort_values(ascending=False).head(15)
axes[1].barh(lt.index[::-1], lt.values[::-1], color=ACCENT, height=0.55, edgecolor="#0f1117")
axes[1].set_title("Top 15 Non-zero Lasso Features")
axes[1].set_xlabel("|Coefficient|")
axes[1].grid(True, axis="x")
plt.tight_layout()
plt.savefig("6_6_lasso.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()

### 6.7 Ridge Regularization Path — Best Alpha Search

In [ ]:
alphas = np.logspace(-3, 3, 60)
rmses  = []
for a in alphas:
    cv = np.sqrt(-cross_val_score(Ridge(alpha=a), Xtr, y_train, cv=5,
                  scoring="neg_mean_squared_error"))
    rmses.append(cv.mean())
best_alpha = alphas[np.argmin(rmses)]
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogx(alphas, rmses, color=ACCENT, linewidth=2)
ax.fill_between(alphas, rmses, alpha=0.1, color=ACCENT)
ax.axvline(best_alpha, color=ACCENT2, linestyle="--", linewidth=2,
    label=f"Best α = {best_alpha:.3f}")
ax.set_title("Ridge Regularization Path — CV RMSE vs Alpha",
    fontsize=13, fontweight="bold", color="#e8ecf5")
ax.set_xlabel("Alpha (log scale)"); ax.set_ylabel("CV RMSE")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("6_7_ridge_path.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(f"Best Ridge alpha: {best_alpha:.4f}")

### 6.8 Final Best Model (Ridge with Tuned Alpha)

In [ ]:
best     = Ridge(alpha=best_alpha)
best.fit(Xtr, y_train)
yp_best  = best.predict(Xte)
fin_rmse = np.sqrt(mean_squared_error(y_test, yp_best))
fin_r2   = r2_score(y_test, yp_best)
fin_mae  = mean_absolute_error(y_test, yp_best)
print("=" * 55)
print(f"  FINAL MODEL: Ridge (alpha={best_alpha:.4f})")
print(f"  Test RMSE : {fin_rmse:.5f}")
print(f"  Test R2   : {fin_r2:.5f}")
print(f"  Test MAE  : {fin_mae:.5f}")
print("=" * 55)

### 6.9 Top Priority Locations — Final Model Output

In [ ]:
df_c["pred_priority"] = best.predict(scaler.transform(df_c[FEAT_COLS]))
top20 = df_c.nlargest(20, "pred_priority")[
    ["location_id","zone","road_condition","road_type",
     "total_incidents","critical_count","total_complaints","pred_priority"]]\
    .reset_index(drop=True)
fig, ax = plt.subplots(figsize=(14, 7))
cols_top = cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top20)))
bars = ax.barh(top20["location_id"][::-1], top20["pred_priority"][::-1],
    color=cols_top, height=0.65, edgecolor="#0f1117")
ax.set_title("Top 20 Highest Priority Locations (Model Predicted)",
    fontsize=14, fontweight="bold", color="#e8ecf5")
ax.set_xlabel("Predicted Priority Score")
for b, (_, row) in zip(bars[::-1], top20.iterrows()):
    ax.text(b.get_width() + 0.002, b.get_y() + b.get_height()/2,
        f"{row['zone']} | {row['road_condition']} | {int(row['critical_count'])} crit",
        va="center", fontsize=7.5, color="#c8cdd8")
ax.set_xlim(0, top20["pred_priority"].max() * 1.35)
ax.grid(True, axis="x")
plt.tight_layout()
plt.savefig("6_9_top_locations.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(top20.to_string(index=False))

---
## 📋 Section 7 — Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 14), facecolor="#0f1117")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)
fig.text(0.5, 0.97,
    "CIVICPULSE SIGHT — WHERE SHOULD WE FIX FIRST?  |  Model Summary Dashboard",
    ha="center", fontsize=17, fontweight="bold", color="#e8ecf5")

# KPI cards
kax = fig.add_subplot(gs[0, :])
kax.set_xlim(0, 1); kax.set_ylim(0, 1); kax.axis("off")
kpis = [
    ("Total Locations",  f"{len(df_c):,}",       ACCENT),
    ("Total Incidents",  f"{len(incidents):,}",   ACCENT2),
    ("Total Complaints", f"{len(complaints):,}",  ACCENT3),
    ("Features Used",   f"{len(FEAT_COLS)}",      ACCENT4),
    ("Best Model R2",   f"{fin_r2:.4f}",          "#a78bfa"),
    ("Test RMSE",       f"{fin_rmse:.5f}",        "#34d399"),
]
for i, (label, value, color) in enumerate(kpis):
    x = 0.08 + i * 0.155
    r = mpatches.FancyBboxPatch((x-0.065, 0.15), 0.13, 0.7,
        boxstyle="round,pad=0.02", linewidth=1.5,
        edgecolor=color, facecolor="#1a1d2e")
    kax.add_patch(r)
    kax.text(x, 0.68, value, ha="center", va="center",
        fontsize=18, fontweight="bold", color=color)
    kax.text(x, 0.3,  label, ha="center", va="center",
        fontsize=9, color="#8890a8")

# CV RMSE bar
ax1 = fig.add_subplot(gs[1, 0:2])
ax1.set_facecolor("#1a1d2e")
ax1.bar(MN, [results[m]["cv_mean"] for m in MN],
    color=mc, width=0.5, edgecolor="#0f1117")
ax1.set_title("CV RMSE by Model", color="#e8ecf5")
ax1.set_ylabel("RMSE")
ax1.tick_params(axis="x", rotation=20, labelsize=8)
ax1.grid(True, axis="y")

# Geo heatmap
ax2 = fig.add_subplot(gs[1, 2:4])
ax2.set_facecolor("#1a1d2e")
sc = ax2.scatter(df_c["longitude"], df_c["latitude"],
    c=df_c["pred_priority"], cmap=CMAP_HOT, s=20, alpha=0.8)
plt.colorbar(sc, ax=ax2, label="Predicted Priority")
ax2.set_title("Geographic Priority Heatmap", color="#e8ecf5")

# Top-10 priority bar
ax3 = fig.add_subplot(gs[2, 0:2])
ax3.set_facecolor("#1a1d2e")
t10 = df_c.nlargest(10, "pred_priority")
ax3.barh(t10["location_id"][::-1], t10["pred_priority"][::-1],
    color=ACCENT2, height=0.6, edgecolor="#0f1117")
ax3.set_title("Top 10 Priority Locations", color="#e8ecf5")
ax3.set_xlabel("Predicted Priority Score")
ax3.grid(True, axis="x")

# Feature importance
ax4 = fig.add_subplot(gs[2, 2:4])
ax4.set_facecolor("#1a1d2e")
tf = pd.Series(np.abs(best.coef_), index=FEAT_COLS)\
       .sort_values(ascending=False).head(10)
ax4.barh(tf.index[::-1], tf.values[::-1],
    color=ACCENT, height=0.55, edgecolor="#0f1117")
ax4.set_title("Top 10 Features (Ridge |coef|)", color="#e8ecf5")
ax4.set_xlabel("|Coefficient|")
ax4.grid(True, axis="x")

plt.savefig("7_summary_dashboard.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Pipeline complete!")

---
## ✅ Section 8 — Conclusions

| Step | Finding |
|------|---------|
| **Data** | 3 clean CSVs, no missing values, joined by `location_id` |
| **Target** | Composite `priority_score` — weighted blend of severity, frequency, complaints, resolution |
| **Outliers** | ~5–8% rows flagged by Z-score; Winsorized at 1–99% to retain high-priority locations |
| **Encoding** | Ordinal for road condition & pedestrian activity; One-hot for zone & road type |
| **Scaling** | `RobustScaler` — median/IQR based, better than StandardScaler when residual outliers exist |
| **Models** | Ridge ≈ ElasticNet >> Lasso > OLS on R² and RMSE |
| **Best model** | Ridge with tuned alpha via regularization path search |
| **Top features** | `weighted_severity`, `total_incidents`, `critical_count`, `weighted_priority` |
| **Explainability** | Coefficient plots + regularization path = full transparency for civic stakeholders |

> **CivicPulse Sight Recommendation**: Locations with Critical road condition + ≥5 critical-severity incidents + High-priority open complaints should be flagged for immediate intervention.